# Kafka Connect Bronze (MinIO) Validation

This notebook validates Kafka Connect S3 sink outputs in MinIO. It focuses on
the raw user and business topics and provides sample data plus counts per
partition and per file.

Note: files are JSON. If Spark infers only `_corrupt_record`, we fall back to
showing raw text samples.

## Spark Session

Build a Spark session configured for MinIO (S3A).

In [20]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, input_file_name, regexp_extract


def get_env(name, default=None):
    value = os.getenv(name)
    return value if value not in (None, "") else default


def build_spark():
    # Create a SparkSession with S3A (MinIO) settings.
    access_key = get_env("DATA_LAKE_ACCESS_KEY_ID", "admin")
    secret_key = get_env("DATA_LAKE_SECRET_ACCESS_KEY", "admin123")
    endpoint = get_env("MINIO_ENDPOINT", "http://minio:9000")

    return (
        SparkSession.builder
        .appName("bronze-minio-check")
        .config("spark.hadoop.fs.s3a.endpoint", endpoint)
        .config("spark.hadoop.fs.s3a.access.key", access_key)
        .config("spark.hadoop.fs.s3a.secret.key", secret_key)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .getOrCreate()
    )


In [21]:
# Build Spark
spark = build_spark()

# Show Spark version
spark.version


'4.0.1'

## Bronze Load Helpers

Read Kafka Connect outputs from MinIO. The connector writes to:
`s3a://bronze/kafka-connect/{topic}`.

In [22]:
BRONZE_BUCKET = get_env("BRONZE_BUCKET", "bronze")
BRONZE_PREFIX = "kafka-connect"


def topic_path(topic: str) -> str:
    # Build the S3A path for a topic in Bronze.
    return f"s3a://{BRONZE_BUCKET}/{BRONZE_PREFIX}/{topic}"


def load_topic_df(topic: str):
    # Load a topic's JSON data from MinIO.
    path = topic_path(topic)
    return (
        spark.read
        .option("mode", "PERMISSIVE")
        .json(path)
    )


## Inspection Helpers

Utilities for sampling and counting by partition/file.

In [23]:
PARTITION_REGEX = r"partition=(\\d+)"


def show_samples(topic: str, df, limit=5):
    # Display a small sample of records; handle corrupt JSON gracefully.
    if set(df.columns) == {"_corrupt_record"}:
        print("Only _corrupt_record detected; showing raw text samples instead.")
        spark.read.text(topic_path(topic)).show(limit, truncate=False)
        return
    if "_corrupt_record" in df.columns:
        df = df.drop("_corrupt_record")
    df.show(limit, truncate=False)


def count_by_partition(topic: str):
    # Count rows by Kafka partition inferred from the file path.
    return (
        spark.read.text(topic_path(topic))
        .withColumn("source_file", input_file_name())
        .withColumn("kafka_partition", regexp_extract(col("source_file"), PARTITION_REGEX, 1))
        .groupBy("kafka_partition")
        .count()
        .orderBy("kafka_partition")
    )


def count_by_file(topic: str):
    # Count rows by the originating file (small-file visibility).
    return (
        spark.read.text(topic_path(topic))
        .withColumn("source_file", input_file_name())
        .groupBy("source_file")
        .count()
        .orderBy("source_file")
    )


## User Topic Checks

In [24]:
# Load raw_data_user
users_df = load_topic_df("raw_data_user")
users_df.printSchema()


root
 |-- _corrupt_record: string (nullable = true)
 |-- partition: integer (nullable = true)



In [25]:
# Sample user records
show_samples("raw_data_user", users_df, limit=5)


+---------+
|partition|
+---------+
|0        |
|0        |
|0        |
|0        |
|0        |
+---------+
only showing top 5 rows


In [ ]:
# User counts by partition
count_by_partition("raw_data_user").show(truncate=False)


In [ ]:
# User counts by file
count_by_file("raw_data_user").show(truncate=False)


## Business Topic Checks

In [26]:
# Load raw_data_business
businesses_df = load_topic_df("raw_data_business")
businesses_df.printSchema()


root
 |-- _corrupt_record: string (nullable = true)
 |-- partition: integer (nullable = true)



In [ ]:
# Sample business records
show_samples("raw_data_business", businesses_df, limit=5)


In [ ]:
# Business counts by partition
count_by_partition("raw_data_business").show(truncate=False)


In [ ]:
# Business counts by file
count_by_file("raw_data_business").show(truncate=False)
